# CN 数据质量专项检查\n\n检查范围：\n1. CSV 源数据完整性（OHLCV 各字段）\n2. Label 质量（10 天 forward return）\n3. Qlib 特征数据可用性\n4. 数据缺失诊断 & 自动修复建议\n\n运行方式：按顺序执行每个 Cell

In [ ]:
import pandas as pd, numpy as np\nfrom pathlib import Path\nfrom datetime import datetime\nimport matplotlib.pyplot as plt\nimport matplotlib; matplotlib.use(\"Agg\")\nROOT = Path.cwd()\nwhile not (ROOT / \"src\").exists() and ROOT != ROOT.parent:\n    ROOT = ROOT.parent\nCSV_DIR = ROOT / \"data\" / \"csv_source\"\nprint(f\"CSV directory: {CSV_DIR}\")\nprint(f\"Files found: {len(list(CSV_DIR.glob('*.csv')))}\")

## 1. CSV 源数据完整性

In [ ]:
def inspect_all_csvs():\n    results = []\n    for f in sorted(CSV_DIR.glob(\"*.csv\")):\n        sym = f.stem\n        try:\n            df = pd.read_csv(f, parse_dates=[\"date\"], low_memory=False)\n            n = len(df)\n            d_min = df[\"date\"].min()\n            d_max = df[\"date\"].max()\n            # Check required columns\n            cols_ok = all(c in df.columns for c in [\"open\",\"high\",\"low\",\"close\",\"volume\"])\n            # Check data quality\n            nan_pct = df[[\"open\",\"high\",\"low\",\"close\"]].isna().mean().mean()\n            # Check OHLC validity\n            high_ge_open = (df[\"high\"] >= df[\"open\"]).fillna(True).all()\n            low_le_open = (df[\"low\"] <= df[\"open\"]).fillna(True).all()\n            # Compute label quality\n            df[\"ret\"] = df[\"close\"].shift(-10) / df[\"close\"] - 1\n            valid_label = df[\"ret\"].notna() & (df[\"ret\"].abs() > 1e-10)\n            label_pct = valid_label.sum() / max(n, 1) * 100\n            # Identify market\n            mkt = \"CN\" if sym.isdigit() else \"US\" if sym.isalpha() else \"OTHER\"\n            results.append({\n                \"symbol\": sym, \"market\": mkt, \"rows\": n,\n                \"start\": d_min, \"end\": d_max,\n                \"cols_ok\": cols_ok, \"nan_pct\": nan_pct,\n                \"ohlc_valid\": high_ge_open and low_le_open,\n                \"label_pct\": label_pct,\n            })\n        except Exception as e:\n            results.append({\"symbol\": sym, \"error\": str(e)})\n    return pd.DataFrame(results)\n\ndf_results = inspect_all_csvs()\nprint(f\"Total symbols: {len(df_results)}\")\nprint(f\"Columns OK: {df_results[df_results['cols_ok']==True].shape[0] if 'cols_ok' in df_results.columns else 'N/A'}\")\nprint(f\"Errors: {df_results['error'].notna().sum() if 'error' in df_results.columns else 0}\")\ndf_results.head(10)

In [ ]:
# Summary by market\nif 'market' in df_results.columns:\n    for mkt in ['CN','US','OTHER']:\n        sub = df_results[df_results['market']==mkt]\n        if len(sub)>0:\n            print(f\"{mkt}: {len(sub)} symbols  rows={sub['rows'].sum():,}  avg_label%={sub['label_pct'].mean():.1f}%  ohlc_ok={sub[sub['ohlc_valid']==True].shape[0]}\")

## 2. Label 质量排名

In [ ]:
cn = df_results[df_results['market']=='CN'].sort_values('label_pct', ascending=False)\nprint(f\"CN symbols: {len(cn)}\")\nprint(f\"100% clean: {(cn['label_pct']>=99.9).sum()}\")\nprint(f\"99%+: {(cn['label_pct']>=99.0).sum()}\")\nprint(f\"95%+: {(cn['label_pct']>=95.0).sum()}\")\nprint(f\"<80%: {(cn['label_pct']<80).sum()}\")\nprint()\nprint(\"Top 20 (best label quality):\")\nfor _,r in cn.head(20).iterrows():\n    print(f\"  {r['symbol']:8s}  {r['rows']:5d} rows  {r['start'].strftime('%Y-%m-%d')} -> {r['end'].strftime('%Y-%m-%d')}  label={r['label_pct']:.1f}%\" )\nprint()\nprint(\"Bottom 20 (worst label quality):\")\nfor _,r in cn.tail(20).iterrows():\n    print(f\"  {r['symbol']:8s}  {r['rows']:5d} rows  {r['start'].strftime('%Y-%m-%d')} -> {r['end'].strftime('%Y-%m-%d')}  label={r['label_pct']:.1f}%\" )

In [ ]:
# Label quality distribution chart\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\naxes[0].hist(cn['label_pct'], bins=50, color='#6366f1', edgecolor='white', alpha=0.8)\naxes[0].axvline(99.0, color='green', ls='--', label='99% threshold')\naxes[0].axvline(95.0, color='orange', ls='--', label='95% threshold')\naxes[0].set_title('CN Label Quality Distribution', fontweight='bold')\naxes[0].set_xlabel('Valid Label %')\naxes[0].legend()\n# OHLC validity\nvalid_ohlc = cn['ohlc_valid'].sum()\ninvalid_ohlc = len(cn) - valid_ohlc\naxes[1].pie([valid_ohlc, invalid_ohlc], labels=[f'Valid ({valid_ohlc})', f'Invalid ({invalid_ohlc})'],\n            colors=['#22c55e','#ef4444'], autopct='%1.1f%%', startangle=90)\naxes[1].set_title('OHLC Validity', fontweight='bold')\nplt.tight_layout(); plt.savefig(ROOT/'notebooks'/'data_quality_viz.png', dpi=100); plt.close()\nprint('Saved: data_quality_viz.png')

## 3. 数据缺失诊断

In [ ]:
# Diagnose WHY labels fail for worst symbols\nworst_sym = cn.iloc[-1]['symbol']\nprint(f\"Diagnosing worst symbol: {worst_sym}\")\ndf = pd.read_csv(CSV_DIR / f'{worst_sym}.csv', parse_dates=['date']).sort_values('date')\nprint(f\"Rows: {len(df)}\")\nprint(f\"Date range: {df['date'].min()} -> {df['date'].max()}\")\nprint(f\"NaN counts: open={df['open'].isna().sum()} high={df['high'].isna().sum()} low={df['low'].isna().sum()} close={df['close'].isna().sum()}\")\n# Check date gaps\ndf['gap'] = df['date'].diff().dt.days\nlarge_gaps = df[df['gap'] > 5]\nprint(f\"Large gaps (>5 days): {len(large_gaps)}\")\nif len(large_gaps) > 0:\n    print(large_gaps[['date','gap']].head(10).to_string())\n# Check label\ndf['ret'] = df['close'].shift(-10) / df['close'] - 1\nprint(f\"Label NaN: {df['ret'].isna().sum()}  Zero: {(df['ret'].abs()<1e-10).sum()}\")\nprint(f\"Label valid: {df['ret'].notna().sum() - (df['ret'].abs()<1e-10).sum()}\")

## 4. OHLC 数据健康检查

In [ ]:
# Check for OHLC anomalies in CN symbols\nprint(\"OHLC anomalies (high<open or low>open):\")\nfor _, r in cn.iterrows():\n    if not r['ohlc_valid']:\n        sym = r['symbol']\n        try:\n            df = pd.read_csv(CSV_DIR / f'{sym}.csv', parse_dates=['date'])\n            hi_lo = (df['high'] < df['open']).sum()\n            lo_hi = (df['low'] > df['open']).sum()\n            print(f\"  {sym}: high<open={hi_lo} rows  low>open={lo_hi} rows  ({r['label_pct']:.1f}% label)\")\n        except Exception as e:\n            print(f\"  {sym}: ERROR {e}\")\n\n# Detailed look at one bad symbol\nbad_syms = cn[cn['ohlc_valid']==False]['symbol'].tolist()\nif bad_syms:\n    sym = bad_syms[0]\n    df = pd.read_csv(CSV_DIR / f'{sym}.csv', parse_dates=['date'])\n    df_hi_ok = df['high'] >= df['open']\n    df_lo_ok = df['low'] <= df['open']\n    bad_rows = df[~(df_hi_ok & df_lo_ok)]\n    print(f\"\\n{sym}: {len(bad_rows)} bad rows out of {len(df)}\")\n    if len(bad_rows) > 0:\n        print(bad_rows[['date','open','high','low','close']].head(10).to_string())

## 5. Qlib 数据可用性

In [ ]:
import sys; sys.path.insert(0, str(ROOT))\nfrom src.common.qlib_init import build_qlib_init_cfg, safe_qlib_init\nfrom qlib.data import D\n\nsafe_qlib_init(build_qlib_init_cfg(None, market='cn'))\n\n# Check Qlib data for top 10 symbols\ntest_syms = cn.head(10)['symbol'].tolist()\nprint(f\"Checking Qlib data for {len(test_syms)} symbols...\")\nfor sym in test_syms:\n    try:\n        df = D.features([sym], ['\$close','\$open','\$high','\$low','\$volume'], start_time='2021-01-01', end_time='2026-06-25')\n        n = len(df)\n        last = df.index.get_level_values('datetime').max()\n        print(f\"  {sym}: {n:4d} rows  last={str(last)[:10]}\")\n    except Exception as e:\n        print(f\"  {sym}: ERROR {e}\")\n\n# Check instruments file\ninstr_path = ROOT / 'data' / 'watchlist' / 'instruments' / 'cn.txt'\nif instr_path.exists():\n    instr_syms = [l.split(chr(9))[0] for l in instr_path.read_text().splitlines() if l.strip()]\n    print(f\"\\nCN instruments file: {len(instr_syms)} symbols\")\n    # Check if all have CSV data\n    missing = [s for s in instr_syms if not (CSV_DIR / f'{s}.csv').exists()]\n    print(f\"  Missing CSV: {len(missing)}\")\n    if missing: print(f\"  Symbols: {missing[:10]}\")

## 6. 自动修复建议

In [ ]:
# Auto-generated fix recommendations\nprint(\"=\" * 60)\nprint(\"DATA QUALITY AUDIT — AUTO RECOMMENDATIONS\")\nprint(\"=\" * 60)\nprint()\n\n# 1. Symbols with OHLC violations\nohlc_bad = cn[cn['ohlc_valid']==False]\nif len(ohlc_bad) > 0:\n    print(f\"1. OHLC violations ({len(ohlc_bad)} symbols):\")\n    for _,r in ohlc_bad.iterrows():\n        print(f\"   uv run python scripts/update_data.py --market cn --lookback-days 3650 --full  # re-download {r['symbol']}\")\n    print()\n\n# 2. Low label quality\nlabel_bad = cn[cn['label_pct'] < 85.0]\nif len(label_bad) > 0:\n    print(f\"2. Low label quality < 85% ({len(label_bad)} symbols)\")\n    syms_str = ' '.join(label_bad['symbol'].head(10).tolist())\n    print(f\"   Consider removing from training set\")\n    print(f\"   Worst 10: {syms_str}\")\n    print()\n\n# 3. Best symbols summary\nbest = cn[cn['label_pct'] >= 99.0]\nprint(f\"3. Training-ready symbols (label >= 99%): {len(best)}\")\nprint(f\"   Best 100 valid% range: {best.head(100)['label_pct'].min():.1f}% - {best.head(1)['label_pct'].values[0]:.1f}%\")\nprint()\n\n# 4. Overall assessment\noverall = cn['label_pct'].mean()\nif overall >= 98:\n    print(f\"4. OVERALL: HEALTHY ({overall:.1f}% avg label quality)\")\nelif overall >= 90:\n    print(f\"4. OVERALL: FAIR ({overall:.1f}% avg label quality) — clean up worst symbols\")\nelse:\n    print(f\"4. OVERALL: POOR ({overall:.1f}% avg label quality) — need data fix\")\n\n# Save report\nreport = {\n    'generated_at': datetime.now().isoformat(),\n    'total_symbols': len(cn),\n    'avg_label_quality': float(cn['label_pct'].mean()),\n    'symbols_99pct': int((cn['label_pct']>=99).sum()),\n    'symbols_95pct': int((cn['label_pct']>=95).sum()),\n    'ohlc_invalid': len(ohlc_bad),\n    'recommended_training_symbols': best.head(100)['symbol'].tolist(),\n}\nimport json\n(ROOT/'artifacts'/'dashboard'/'data_quality_report.json').write_text(json.dumps(report, indent=2, default=str))\nprint(\"\\nReport saved: artifacts/dashboard/data_quality_report.json\")